# HK Free Press News RAG Ingestion with LlamaIndex and ChromaDB

This notebook builds the ingestion side of your news chatbot. It reads text files from `data/hk_free_press_news`, cleans and normalizes each article, splits the articles into retrieval-friendly chunks, embeds those chunks with a local Hugging Face model, and stores them in a persistent ChromaDB collection named `news_chat`.

The flow is:

1. Install and import the required libraries.
2. Point the notebook at your text news folder and ChromaDB folder.
3. Load `.txt` news articles as LlamaIndex documents with useful metadata.
4. Chunk each article into retrieval-friendly nodes.
5. Insert the nodes into ChromaDB.
6. Run a small retrieval test to confirm the vector database is working.

In [ ]:
# Install the RAG ingestion dependencies in the active notebook kernel.
# Run this cell once, then restart the kernel if imports still fail.
%pip install -U llama-index llama-index-vector-stores-chroma llama-index-embeddings-huggingface chromadb sentence-transformers

In [1]:
import re
import unicodedata
from pathlib import Path

import chromadb
from chromadb.errors import NotFoundError
from llama_index.core import Settings, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document, TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

## 1. Configure Paths and Embeddings

The news folder is your source corpus. Each `.txt` file is treated as one news article before chunking.

The ChromaDB folder is where the vector database is saved. The collection name is now `news_chat`, so this dataset stays separate from your previous `mini_wikipedia` collection.

This notebook uses `BAAI/bge-small-en-v1.5` because it is a strong local embedding model and does not require an API key.

In [2]:
PROJECT_ROOT = Path(r"C:\Program Files\Studying\coding\RAG_project")
NEWS_DIR = PROJECT_ROOT / "data" / "hk_free_press_news"
CHROMA_DIR = PROJECT_ROOT / "chromadb_store"
COLLECTION_NAME = "news_chat"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"

if not NEWS_DIR.exists():
    raise FileNotFoundError(f"News text folder not found: {NEWS_DIR}")

# LlamaIndex uses Settings as global defaults for later index and query operations.
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)
Settings.llm = None

print(f"News source folder: {NEWS_DIR}")
print(f"ChromaDB folder: {CHROMA_DIR}")
print(f"Collection name: {COLLECTION_NAME}")
print(f"Embedding model: {EMBED_MODEL_NAME}")

LLM is explicitly disabled. Using MockLLM.
News source folder: C:\Program Files\Studying\coding\RAG_project\data\hk_free_press_news
ChromaDB folder: C:\Program Files\Studying\coding\RAG_project\chromadb_store
Collection name: news_chat
Embedding model: BAAI/bge-small-en-v1.5


## 2. Load Text News Articles

Your new dataset is already stored as text files, so we do not need PDF extraction anymore. That is a good improvement for RAG: plain text is usually cleaner, faster to load, and less likely to produce unreadable characters than PDF parsing.

In this section, each `.txt` file becomes one LlamaIndex `Document`. The loader also extracts useful metadata from the file name:

- `article_date`: the date prefix at the start of the file name, such as `01-01-2026`
- `article_title`: the article title after the underscore
- `file_name` and `file_path`: the original source file information

This metadata is stored alongside each chunk in ChromaDB, so later retrieval can show where an answer came from.

In [3]:
def list_text_files(news_dir: Path) -> list[Path]:
    """Return all text news files under the source folder."""
    # Use rglob so nested folders are included automatically if you organize files later.
    return sorted(news_dir.rglob("*.txt"))


def normalize_news_text(text: str) -> str:
    """Clean text-file artifacts while preserving the article content."""
    # Unicode normalization standardizes curly quotes, full-width characters, and ligatures.
    cleaned_text = unicodedata.normalize("NFKC", text or "")

    # Remove invisible or problematic characters that can hurt chunking and retrieval.
    cleaned_text = cleaned_text.replace("\x00", " ").replace("\ufeff", "").replace("\u00ad", "")

    # Normalize line endings first, then collapse excessive spaces and blank lines.
    cleaned_text = re.sub(r"\r\n?", "\n", cleaned_text)
    cleaned_text = re.sub(r"[ \t]+", " ", cleaned_text)
    cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text)

    return cleaned_text.strip()


def parse_news_file_metadata(file_path: Path) -> dict[str, str]:
    """Extract article metadata from the HKFP text file name."""
    # File names look like: 01-01-2026_Article title.txt
    # Keeping the date and title as metadata helps retrieval results act like citations.
    file_stem = file_path.stem
    date_part, separator, title_part = file_stem.partition("_")

    article_date = date_part if separator else "unknown_date"
    article_title = title_part if separator else file_stem

    return {
        "source_type": "hk_free_press_news_txt",
        "article_date": article_date,
        "article_title": article_title,
        "file_name": file_path.name,
        "file_path": str(file_path),
    }


def read_text_file(file_path: Path) -> str:
    """Read one news text file with a small encoding fallback."""
    # Most modern scraped text is UTF-8, but utf-8-sig also removes a BOM if present.
    for encoding in ("utf-8-sig", "utf-8", "cp1252"):
        try:
            return file_path.read_text(encoding=encoding)
        except UnicodeDecodeError:
            continue

    # If all strict decoders fail, replace broken characters instead of stopping ingestion.
    return file_path.read_text(encoding="utf-8", errors="replace")


def load_news_documents(news_dir: Path) -> list[Document]:
    """Load text news articles into LlamaIndex Documents with cleaned text and metadata."""
    documents: list[Document] = []
    skipped_files: list[str] = []

    for text_path in list_text_files(news_dir):
        raw_text = read_text_file(text_path)
        cleaned_text = normalize_news_text(raw_text)

        # Empty files should not become chunks because they cannot help retrieval.
        if not cleaned_text:
            skipped_files.append(text_path.name)
            continue

        # The Document is the article-level object that LlamaIndex will later split into chunks.
        documents.append(
            Document(
                text=cleaned_text,
                metadata=parse_news_file_metadata(text_path),
            )
        )

    if skipped_files:
        print("Warning: some text files were empty after cleaning and were skipped.")
        for file_name in skipped_files[:10]:
            print(f"- {file_name}")
        if len(skipped_files) > 10:
            print(f"... and {len(skipped_files) - 10} more skipped file(s)")

    return documents


def preview_news_documents(documents: list[Document], preview_count: int = 3) -> None:
    """Print a few loaded articles so you can quickly judge text quality."""
    for index, document in enumerate(documents[:preview_count], start=1):
        metadata = document.metadata
        print(f"\n--- Preview {index} ---")
        print(f"Date: {metadata['article_date']}")
        print(f"Title: {metadata['article_title']}")
        print(document.text[:1200])


text_files = list_text_files(NEWS_DIR)
print(f"Found {len(text_files)} text news file(s).")
for text_file in text_files[:10]:
    print(f"- {text_file.name}")

if len(text_files) > 10:
    print(f"... and {len(text_files) - 10} more")


documents = load_news_documents(NEWS_DIR)
print(f"Loaded {len(documents)} news article document(s).")
preview_news_documents(documents)

Found 1022 text news file(s).
- 01-01-2026_Chinese homeschooled students embrace freer youth in cutthroat market.txt
- 01-01-2026_Explainer How deadly Tai Po fire brings to light bid-rigging epidemic in Hong Kong renovation industry.txt
- 01-01-2026_Listed companies face a new year’s carbon resolution – let’s join them.txt
- 01-01-2026_Taiwan’s president vows to defend sovereignty after China drills.txt
- 01-02-2026_Chinese cash in jewellery at automated gold recyclers as prices soar.txt
- 01-02-2026_UK Prime Minister Keir Starmer ends China trip aimed at reset despite Trump warning.txt
- 01-02-2026_‘Is there a choice or no choice’ Gov’t handling of long-term housing frustrates Tai Po fire survivors.txt
- 01-03-2026_Iran’s Ayatollah Ali Khamenei killed in massive US and Israeli attack.txt
- 01-03-2026_Tai Po fire Some survivors wary over resettlement plan after ‘ignored’ calls to rebuild on site.txt
- 01-03-2026_‘Unofficial’ talks on plastic pollution treaty to begin in Japan.txt
... a

## 3. Chunk the News Text

RAG works better when articles are split into chunks that are large enough to contain meaning but small enough to retrieve precisely.

Here we use sentence-aware chunking with overlap. The overlap is important for news articles because names, places, and quoted context may span chunk boundaries. Each chunk keeps the article metadata from the original text file, so retrieved chunks can still show the article date, title, and file name.

In [4]:
def build_text_nodes(
    documents: list[Document],
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> list[TextNode]:
    """Split loaded news documents into chunks that can be embedded and retrieved."""
    # SentenceSplitter tries to preserve sentence boundaries instead of cutting text blindly.
    splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    nodes = splitter.get_nodes_from_documents(documents)

    # Store chunk-level metadata to make retrieved news results easier to inspect later.
    for chunk_number, node in enumerate(nodes, start=1):
        node.metadata["chunk_number"] = chunk_number
        node.metadata["chunk_size"] = chunk_size
        node.metadata["chunk_overlap"] = chunk_overlap

    return nodes


nodes = build_text_nodes(documents)

print(f"Loaded {len(documents)} news article document(s).")
print(f"Created {len(nodes)} text chunk(s).")
print("\nPreview of first chunk:")
print(nodes[0].get_content()[:800] if nodes else "No chunks created.")

Loaded 1020 news article document(s).
Created 1499 text chunk(s).

Preview of first chunk:
By Mary Yang

Fourteen-year-old Estella spends her weekdays studying Spanish, rock climbing or learning acupuncture in her living room as part of her homeschooling since she left China’s gruelling public school system.

Her parents withdrew her from her Shanghai school three years ago, worried she was struggling to keep up with a demanding curriculum they believe will soon be outdated in the era of artificial intelligence (AI).

They are among a small number of parents in China who are rethinking the country’s rigorous education system, in which school days can last 10 hours, with students often working late into the evening on extra tutoring and homework.

“In the future, education models and jobs will face huge changes due to AI,” Estella’s mother Xu Zoe told AFP, using a pseudonym.

“We


In [5]:
nodes[0].get_content() if len(nodes) > 0 else "No chunk at index 0"

'By Mary Yang\n\nFourteen-year-old Estella spends her weekdays studying Spanish, rock climbing or learning acupuncture in her living room as part of her homeschooling since she left China’s gruelling public school system.\n\nHer parents withdrew her from her Shanghai school three years ago, worried she was struggling to keep up with a demanding curriculum they believe will soon be outdated in the era of artificial intelligence (AI).\n\nThey are among a small number of parents in China who are rethinking the country’s rigorous education system, in which school days can last 10 hours, with students often working late into the evening on extra tutoring and homework.\n\n“In the future, education models and jobs will face huge changes due to AI,” Estella’s mother Xu Zoe told AFP, using a pseudonym.\n\n“We wanted to get used to the uncertainty early.”\n\nHomeschooling is banned in China, although authorities generally overlook rare individual cases.\n\nJust 6,000 Chinese children were homesc

In [6]:
nodes[1].get_content() if len(nodes) > 1 else "No chunk at index 1"

'One mother in Zhejiang province, who wished to remain unidentified for fear of repercussions, said she used an AI chatbot to create a lesson plan on recycling for her nine-year-old homeschooled son.\n\n“The development of AI has allowed me to say that what you learn in a classroom, you don’t need anymore,” she told AFP.\n\nHer son studies Chinese and maths using coursework from his former public school in the mornings and spends afternoons working on projects or outdoor activities.\n\nHowever, his mother, a former teacher, plans to re-enrol her son when he reaches middle school.\n\n“There’s no way to meet his social needs at home,” she said.\n\n‘Don’t be afraid’\n\nTime with children her age was one of the biggest losses for 24-year-old Gong Yimei, whose father pulled her out of school at age eight to focus on art.\n\nShe studied on her own with few teachers, and most of the people she called friends were twice her age.\n\nBut at home, Gong told AFP she had more free time to consider 

In [7]:
nodes[2].get_content() if len(nodes) > 2 else "No chunk at index 2"

'More than a year before a massive fire tore through Wang Fuk Court, an eight-block housing estate in Tai Po, residents started seeing red flags about the renovation project.\n\nThey shared tender documents and conviction records on Facebook groups, pointing to what they believed was a complex web of corruption and greed. They complained to the Labour Department and the Independent Commission Against Corruption (ICAC), to no avail.\n\nThe blaze in late November, which killed at least 161 people, not only highlighted gaps in safety oversight but also laid bare corruption and bid-rigging in Hong Kong’s building renovation industry.\n\nIn the aftermath, Chief Executive John Lee established an independent committee to probe the cause of the fire and other issues, including potential bid-rigging and corruption.\n\nPrestige Construction & Engineering Company, the maintenance contractor at Wang Fuk Court, and Will Power Architects, the consultant overseeing the project, are now under investig

In [8]:
for index, node in enumerate(nodes[:3], start=1):
    print(f"Chunk {index} length: {len(node.get_content())}")

if not nodes:
    print("No chunks were created.")

Chunk 1 length: 3416
Chunk 2 length: 1527
Chunk 3 length: 3177


## 4. Insert Chunks into ChromaDB

This cell creates a persistent ChromaDB collection and inserts the chunks through LlamaIndex. `RESET_COLLECTION = True` rebuilds this notebook's collection cleanly, which prevents duplicate chunks when you rerun the notebook. Set it to `False` only when you intentionally want to append new chunks.

In [9]:
RESET_COLLECTION = True


def get_chroma_collection(
    chroma_dir: Path,
    collection_name: str,
    reset_collection: bool = False,
):
    """Create or load a persistent ChromaDB collection."""
    # PersistentClient writes vectors and metadata to disk instead of keeping them in memory.
    chroma_dir.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(chroma_dir))

    # Some ChromaDB versions raise NotFoundError when the collection is missing,
    # while older versions may raise ValueError. Catch both so reset stays safe.
    if reset_collection:
        try:
            client.delete_collection(collection_name)
            print(f"Deleted existing collection: {collection_name}")
        except (NotFoundError, ValueError):
            print(f"Collection did not exist yet: {collection_name}")

    return client.get_or_create_collection(collection_name)


def build_or_update_vector_index(
    text_nodes: list[TextNode],
    chroma_dir: Path = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME,
    reset_collection: bool = RESET_COLLECTION,
) -> VectorStoreIndex:
    """Embed nodes and store them in ChromaDB through LlamaIndex."""
    # ChromaVectorStore adapts the Chroma collection to LlamaIndex's vector store interface.
    collection = get_chroma_collection(chroma_dir, collection_name, reset_collection)
    vector_store = ChromaVectorStore(chroma_collection=collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    # VectorStoreIndex computes embeddings for each node and writes vectors plus metadata.
    return VectorStoreIndex(text_nodes, storage_context=storage_context)


index = build_or_update_vector_index(nodes)
collection = get_chroma_collection(CHROMA_DIR, COLLECTION_NAME)

print(f"Collection name: {COLLECTION_NAME}")
print(f"Stored vector count: {collection.count()}")

Collection did not exist yet: news_chat
Collection name: news_chat
Stored vector count: 1499


## 5. Query the Stored Chunks

Once the chunks are inside ChromaDB, you do not need to insert them again every time you reopen the notebook.

After reopening the notebook, rerun the setup cells so the imports, paths, and embedding model are available, then load the existing Chroma collection from disk and query it. The code cell below supports both cases:

- querying the freshly built index in the current session
- reconnecting to an existing persisted ChromaDB collection after reopening the notebook

In [11]:
def load_existing_index(
    chroma_dir: Path = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME,
) -> VectorStoreIndex:
    """Reconnect to an existing persisted ChromaDB collection without reinserting data."""
    # This is the path you use after closing and reopening the notebook.
    collection = get_chroma_collection(chroma_dir, collection_name, reset_collection=False)
    vector_store = ChromaVectorStore(chroma_collection=collection)
    return VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=Settings.embed_model)


def retrieve_relevant_chunks(
    query: str,
    top_k: int = 5,
    query_index: VectorStoreIndex | None = None,
):
    """Retrieve the most relevant news chunks for a user question."""
    # Use the provided index when one is passed in, otherwise fall back to the current session index.
    active_index = query_index if query_index is not None else index
    retriever = active_index.as_retriever(similarity_top_k=top_k)
    results = retriever.retrieve(query)

    # Print article metadata beside each chunk so you can inspect where answers come from.
    for rank, result in enumerate(results, start=1):
        metadata = result.node.metadata
        article_date = metadata.get("article_date", "unknown date")
        article_title = metadata.get("article_title", metadata.get("file_name", "unknown article"))
        chunk_number = metadata.get("chunk_number", "unknown chunk")

        print(f"\n--- Result {rank} | score={result.score:.4f} ---")
        print(f"Source: {article_date} | {article_title} | chunk: {chunk_number}")
        print(result.node.get_content()[:1200])

    return results


# Use the in-memory index if you just inserted data in this session.
sample_query = "Tai Po fire issue"
retrieved_chunks = retrieve_relevant_chunks(sample_query, top_k=3)


--- Result 1 | score=0.6888 ---
Source: 13-04-2026 | Tai Po blaze probe No standard operation protocols for failed alarms during fire, senior firefighter says | chunk: 688
A fire station officer deployed to the fatal Wang Fuk Court fire has told an independent committee that the Fire Services Department (FSD) has no standard operating procedures to deal with non-functioning alarm systems during a blaze.

Tai Po Fire Station’s Senior Station Officer Hui Kin-on gave testimony on Monday, the 11th day of the hearings investigating the fire in Tai Po that killed 168 people, including a firefighter.

“Would it be a much greater help if there were clear instructions?” asked Lee Shu-wan, a lawyer representing the independent committee.

“Agree,” Hui replied, adding that the operational response would depend on the commanding officer’s orders, based on the actual situation at the scene.

The probe into the Wang Fuk Court fire previously heard testimony from an electrician for estate management

In [ ]:
# If you reopen the notebook later, rerun the setup cells and then use this block.
sample_query = 'Tai Po fire issue'
reloaded_index = load_existing_index()
retrieved_chunks_after_reopen = retrieve_relevant_chunks(sample_query, top_k=3, query_index=reloaded_index)